# Transfer Learning - Moirai-2 v1 to Vietnam

Chien luoc: load pretrained Moirai-2 weights (multi-country) -> fine-tune tren du lieu Viet Nam qua 2 phase.

In [19]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch: {torch.__version__}')
print(f'Device: {DEVICE}')

BASE_DIR = Path('.')
VN_DATA_PATH = BASE_DIR / 'data' / 'processed' / 'VN_data' / 'vn_tft_ready.csv'
PRETRAIN_CFG = BASE_DIR / 'checkpoint' / 'moirai2_v1_config.json'
PRETRAIN_WEIGHTS = BASE_DIR / 'checkpoint' / 'moirai2_v1_pretrain_weights.pt'
CKPT_DIR = BASE_DIR / 'checkpoint'
LOG_DIR = BASE_DIR / 'lightning_logs' / 'moirai2_vn_v1'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

with open(PRETRAIN_CFG, 'r', encoding='utf-8') as f:
    pre_cfg = json.load(f)

# Transfer settings aligned with TFT transfer policy for fair comparison.
CFG = dict(
    seed=42,
    context_length=min(18, int(pre_cfg['context_length'])),
    prediction_length=int(pre_cfg['prediction_length']),
    val_cutoff_months=12,
    d_model=int(pre_cfg['d_model']),
    nhead=int(pre_cfg['nhead']),
    num_layers=int(pre_cfg['num_layers']),
    ff_mult=4,
    dropout=float(pre_cfg['dropout']),
    quantiles=pre_cfg['quantiles'],
    economic_vars=[
        'IPI_Value', 'CPI_Value', 'GDP_trillion', 'Oil_Price',
        'FDI_disbursed', 'gas_price', 'castlecoal_price',
    ],
    phase1_lr=1e-3,
    phase1_epochs=15,
    phase1_patience=8,
    phase2_lr=1e-5,
    phase2_epochs=20,
    phase2_patience=10,
    weight_decay=1e-4,
    batch_size=16,
    gradient_clip_val=0.5,
    num_workers=0,
    mixed_precision=False,
 )

pl.seed_everything(CFG['seed'], workers=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

print('Transfer config:')
for k, v in CFG.items():
    print(f'  {k:<24}: {v}')

Seed set to 42


Torch: 2.5.1+cu121
Device: cuda
Transfer config:
  seed                    : 42
  context_length          : 18
  prediction_length       : 6
  val_cutoff_months       : 12
  d_model                 : 192
  nhead                   : 6
  num_layers              : 4
  ff_mult                 : 4
  dropout                 : 0.15
  quantiles               : [0.1, 0.25, 0.5, 0.75, 0.9]
  economic_vars           : ['IPI_Value', 'CPI_Value', 'GDP_trillion', 'Oil_Price', 'FDI_disbursed', 'gas_price', 'castlecoal_price']
  phase1_lr               : 0.001
  phase1_epochs           : 15
  phase1_patience         : 8
  phase2_lr               : 1e-05
  phase2_epochs           : 20
  phase2_patience         : 10
  weight_decay            : 0.0001
  batch_size              : 16
  gradient_clip_val       : 0.5
  num_workers             : 0
  mixed_precision         : False


In [20]:
df = pd.read_csv(VN_DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['entity', 'series', 'date']).reset_index(drop=True)

keep_series = ['Coal', 'Gas', 'Hydro', 'Solar', 'Wind']
series_lower = df['series'].astype(str).str.strip().str.lower()
drop_mask = series_lower.eq('other fossil') | series_lower.eq('other_fossil')
df = df[~drop_mask].copy()
df = df[df['series'].isin(keep_series)].copy()

if 'prec_zscore' not in df.columns and 'precipitation' in df.columns:
    mu = df.groupby('entity')['precipitation'].transform('mean')
    std = df.groupby('entity')['precipitation'].transform('std').replace(0, 1)
    df['prec_zscore'] = (df['precipitation'] - mu) / std

weather_cols = ['temperature', 'solar', 'humidity', 'precipitation']
for col in weather_cols:
    if col in df.columns:
        df[col] = df.groupby(['entity', 'series'])[col].shift(1)

# Match TFT transfer: lag economic variables and backfill the first timestep after shifting.
existing_economic_vars = [v for v in CFG['economic_vars'] if v in df.columns]
print('Lagging economic vars:', existing_economic_vars)
for var in existing_economic_vars:
    df[var] = df.groupby(['entity', 'series'])[var].shift(1)
if existing_economic_vars:
    df[existing_economic_vars] = df.groupby(['entity', 'series'])[existing_economic_vars].transform(lambda x: x.bfill())

# Interpolate only columns that exist in both VN data and pretrain feature list.
for col in pre_cfg['feature_cols']:
    if col in df.columns:
        df[col] = df.groupby(['entity', 'series'])[col].transform(lambda x: x.interpolate().bfill().ffill())

df['time_idx'] = df.groupby(['entity', 'series'])['date'].rank(method='dense').astype(int) - 1
training_cutoff = int(df['time_idx'].max()) - CFG['val_cutoff_months']

print(f'VN shape: {df.shape}')
print(f"Series: {sorted(df['series'].unique())}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f'Training cutoff time_idx: {training_cutoff}')

Lagging economic vars: ['IPI_Value', 'CPI_Value', 'GDP_trillion', 'Oil_Price', 'FDI_disbursed', 'gas_price', 'castlecoal_price']
VN shape: (360, 31)
Series: ['Coal', 'Gas', 'Hydro', 'Solar', 'Wind']
Date range: 2019-01-01 -> 2024-12-01
Training cutoff time_idx: 59


In [21]:
KNOWN_REAL_CANDIDATES = ['time_idx', 'month', 'month_sin', 'month_cos', 'quarter', 'year']
UNKNOWN_REAL_CANDIDATES = [
    'temperature', 'solar', 'humidity', 'precipitation',
    'log_precipitation', 'prec_zscore', 'temp_anomaly', 'solar_norm',
    'humidity_anomaly', 'prec_lag_1', 'prec_lag_2', 'precip_roll6',
]

feature_candidates = list(dict.fromkeys(
    KNOWN_REAL_CANDIDATES + UNKNOWN_REAL_CANDIDATES + existing_economic_vars
))

# Keep features that exist in VN data and were present in Moirai pretrain.
feature_cols = [c for c in feature_candidates if c in df.columns and c in pre_cfg['feature_cols']]
missing_from_vn = [c for c in feature_candidates if c not in df.columns]
missing_from_pretrain = [c for c in feature_candidates if c in df.columns and c not in pre_cfg['feature_cols']]

print('Feature cols used:', feature_cols)
print('Num features:', len(feature_cols))
if missing_from_vn:
    print('Missing from VN:', missing_from_vn)
if missing_from_pretrain:
    print('Excluded (not in pretrain):', missing_from_pretrain)

target_col = pre_cfg['target_col']
group_cols = pre_cfg['group_cols']

class WindowDataset(Dataset):
    def __init__(self, data, group_cols, feature_cols, target_col, context_length, prediction_length, cutoff_idx=None, split='train'):
        self.samples = []
        grouped = data.groupby(group_cols)

        for _, g in grouped:
            g = g.sort_values('time_idx').reset_index(drop=True)
            x_feat = g[feature_cols].values.astype(np.float32)
            y_val = g[target_col].values.astype(np.float32)
            t_idx = g['time_idx'].values.astype(int)

            max_start = len(g) - context_length - prediction_length
            if max_start < 0:
                continue

            for start in range(max_start + 1):
                enc_start = start
                enc_end = start + context_length
                dec_end = enc_end + prediction_length

                last_encoder_idx = t_idx[enc_end - 1]
                if cutoff_idx is not None:
                    if split == 'train' and last_encoder_idx > cutoff_idx:
                        continue
                    if split == 'val' and last_encoder_idx <= cutoff_idx:
                        continue

                x_hist_feat = x_feat[enc_start:enc_end]
                x_hist_target = y_val[enc_start:enc_end].reshape(-1, 1)
                x_hist = np.concatenate([x_hist_target, x_hist_feat], axis=1)
                y_future = y_val[enc_end:dec_end]
                self.samples.append((x_hist, y_future))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_hist, y_future = self.samples[idx]
        return torch.from_numpy(x_hist), torch.from_numpy(y_future)

train_ds = WindowDataset(
    data=df,
    group_cols=group_cols,
    feature_cols=feature_cols,
    target_col=target_col,
    context_length=CFG['context_length'],
    prediction_length=CFG['prediction_length'],
    cutoff_idx=training_cutoff,
    split='train'
)

val_ds = WindowDataset(
    data=df,
    group_cols=group_cols,
    feature_cols=feature_cols,
    target_col=target_col,
    context_length=CFG['context_length'],
    prediction_length=CFG['prediction_length'],
    cutoff_idx=training_cutoff,
    split='val'
)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False, num_workers=CFG['num_workers'])

x0, y0 = next(iter(train_loader))
print(f'Train samples: {len(train_ds):,}')
print(f'Val samples: {len(val_ds):,}')
print(f'Input shape: {x0.shape}')
print(f'Target shape: {y0.shape}')

Feature cols used: ['time_idx', 'month', 'year', 'temperature', 'solar', 'humidity', 'precipitation']
Num features: 7
Missing from VN: ['month_sin', 'month_cos', 'quarter', 'log_precipitation', 'temp_anomaly', 'solar_norm', 'humidity_anomaly', 'prec_lag_1', 'prec_lag_2']
Excluded (not in pretrain): ['prec_zscore', 'precip_roll6', 'IPI_Value', 'CPI_Value', 'GDP_trillion', 'Oil_Price', 'FDI_disbursed', 'gas_price', 'castlecoal_price']
Train samples: 215
Val samples: 30
Input shape: torch.Size([16, 18, 8])
Target shape: torch.Size([16, 6])


In [22]:
class Moirai2Forecaster(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, ff_mult, dropout, prediction_length, quantiles):
        super().__init__()
        self.prediction_length = prediction_length
        self.quantiles = quantiles
        self.nq = len(quantiles)

        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * ff_mult,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, prediction_length * self.nq)

    def forward(self, x):
        z = self.input_proj(x)
        z = self.encoder(z)
        z = self.norm(z[:, -1, :])
        out = self.head(z)
        return out.view(-1, self.prediction_length, self.nq)

def quantile_loss(pred_q, target, quantiles):
    losses = []
    for i, q in enumerate(quantiles):
        err = target - pred_q[:, :, i]
        losses.append(torch.maximum((q - 1) * err, q * err).unsqueeze(-1))
    return torch.cat(losses, dim=-1).mean()

def compute_metrics(y_true, y_pred):
    mae = torch.mean(torch.abs(y_true - y_pred)).item()
    rmse = torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()
    smape = (200.0 * torch.mean(torch.abs(y_true - y_pred) / (torch.abs(y_true) + torch.abs(y_pred) + 1e-8))).item()
    wape = (100.0 * torch.sum(torch.abs(y_true - y_pred)) / (torch.sum(torch.abs(y_true)) + 1e-8)).item()
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    r2 = (1 - ss_res / (ss_tot + 1e-8)).item()
    return mae, rmse, smape, wape, r2

class LitMoirai2(pl.LightningModule):
    def __init__(self, cfg, input_dim):
        super().__init__()
        self.save_hyperparameters({'cfg': cfg, 'input_dim': input_dim})
        self.cfg = cfg
        self.model = Moirai2Forecaster(
            input_dim=input_dim,
            d_model=cfg['d_model'],
            nhead=cfg['nhead'],
            num_layers=cfg['num_layers'],
            ff_mult=cfg['ff_mult'],
            dropout=cfg['dropout'],
            prediction_length=cfg['prediction_length'],
            quantiles=cfg['quantiles']
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred_q = self(x)
        loss = quantile_loss(pred_q, y, self.cfg['quantiles'])
        self.log('train_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred_q = self(x)
        loss = quantile_loss(pred_q, y, self.cfg['quantiles'])
        median_idx = self.cfg['quantiles'].index(0.5) if 0.5 in self.cfg['quantiles'] else len(self.cfg['quantiles']) // 2
        y_hat = pred_q[:, :, median_idx]
        mae, rmse, smape, wape, r2 = compute_metrics(y, y_hat)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log('val_mae', mae, prog_bar=False)
        self.log('val_rmse', rmse, prog_bar=False)
        self.log('val_smape', smape, prog_bar=False)
        self.log('val_wape', wape, prog_bar=False)
        self.log('val_r2', r2, prog_bar=False)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.cfg['learning_rate'],
            weight_decay=self.cfg['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

def freeze_encoder_for_phase1(model):
    frozen = 0
    total = 0
    for name, p in model.named_parameters():
        total += p.numel()
        # TFT phase-1 keeps heads/trainable components adapting; mimic that behavior.
        if name.startswith('encoder'):
            p.requires_grad = False
            frozen += p.numel()
        else:
            p.requires_grad = True
    return frozen, total

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

In [23]:
input_dim = 1 + len(feature_cols)
transfer_model = LitMoirai2(
    cfg={**CFG, 'learning_rate': CFG['phase1_lr'], 'weight_decay': CFG['weight_decay']},
    input_dim=input_dim
)

state_obj = torch.load(PRETRAIN_WEIGHTS, map_location='cpu')
pre_state = state_obj['model_state_dict']
current_state = transfer_model.model.state_dict()

# Load only parameters with matching names and shapes to avoid size mismatch crashes
loadable_state = {}
skipped_mismatch = []
for name, tensor in pre_state.items():
    if name in current_state and current_state[name].shape == tensor.shape:
        loadable_state[name] = tensor
    else:
        if name in current_state:
            skipped_mismatch.append(
                f"{name}: ckpt {tuple(tensor.shape)} vs model {tuple(current_state[name].shape)}"
            )

missing, unexpected = transfer_model.model.load_state_dict(loadable_state, strict=False)
print('Loaded pretrained weights (shape-safe partial load)')
print('Loaded keys:', len(loadable_state))
print('Missing keys after load:', len(missing))
print('Unexpected keys:', len(unexpected))
print('Skipped due to shape mismatch:', len(skipped_mismatch))
if skipped_mismatch:
    print('Sample mismatched keys:')
    for msg in skipped_mismatch[:5]:
        print('  -', msg)

frozen, total = freeze_encoder_for_phase1(transfer_model.model)
print(f'Phase 1 freeze params: {frozen:,} / {total:,} ({100.0 * frozen / max(total, 1):.1f}%)')

ckpt_p1 = ModelCheckpoint(
    dirpath=str(CKPT_DIR),
    filename='moirai2_vn_phase1_{epoch:02d}_{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=False,
    verbose=True
)
es_p1 = EarlyStopping(monitor='val_loss', patience=CFG['phase1_patience'], mode='min', verbose=True)

trainer_p1 = pl.Trainer(
    accelerator='gpu' if DEVICE == 'cuda' else 'cpu',
    devices=1,
    max_epochs=CFG['phase1_epochs'],
    gradient_clip_val=CFG['gradient_clip_val'],
    callbacks=[ckpt_p1, es_p1],
    logger=TensorBoardLogger(save_dir=str(BASE_DIR / 'lightning_logs'), name='moirai2_vn_phase1'),
    enable_progress_bar=True,
    precision='16-mixed' if (DEVICE == 'cuda' and CFG.get('mixed_precision', False)) else '32'
)

trainer_p1.fit(transfer_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print(f'Phase 1 best val_loss: {ckpt_p1.best_model_score}')
print(f'Phase 1 best ckpt: {ckpt_p1.best_model_path}')

Loaded pretrained weights (shape-safe partial load)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13760\3432389580.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_obj = torch.load(PRETRAIN_WEIGHTS, map_location='cpu')


Loaded keys: 53
Missing keys after load: 1
Unexpected keys: 0
Skipped due to shape mismatch: 1
Sample mismatched keys:
  - input_proj.weight: ckpt (192, 15) vs model (192, 8)
Phase 1 freeze params: 1,779,456 / 1,787,358 (99.6%)


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Moirai2Forecaster │  1.8 M │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 1.8 M                                                                                        
Total params: 1.8 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Epoch 11, global step 168: 'val_loss' was not in top 1


Epoch 12, global step 182: 'val_loss' was not in top 1


Epoch 13, global step 196: 'val_loss' was not in top 1


Epoch 14, global step 210: 'val_loss' was not in top 1
`Trainer.fit` stopped: `max_epochs=15` reached.


Phase 1 best val_loss: 1.2925317287445068
Phase 1 best ckpt: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\moirai2_vn_phase1_epoch=09_val_loss=1.2925.ckpt


In [24]:
phase2_cfg = {**CFG, 'learning_rate': CFG['phase2_lr'], 'weight_decay': CFG['weight_decay']}
model_p2 = LitMoirai2.load_from_checkpoint(str(ckpt_p1.best_model_path), cfg=phase2_cfg, input_dim=input_dim)

unfreeze_all(model_p2.model)

ckpt_p2 = ModelCheckpoint(
    dirpath=str(CKPT_DIR),
    filename='moirai2_vn_v1_{epoch:02d}_{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=False,
    verbose=True
)
es_p2 = EarlyStopping(monitor='val_loss', patience=CFG['phase2_patience'], mode='min', verbose=True)
lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer_p2 = pl.Trainer(
    accelerator='gpu' if DEVICE == 'cuda' else 'cpu',
    devices=1,
    max_epochs=CFG['phase2_epochs'],
    gradient_clip_val=CFG['gradient_clip_val'],
    callbacks=[ckpt_p2, es_p2, lr_monitor],
    logger=TensorBoardLogger(save_dir=str(BASE_DIR / 'lightning_logs'), name='moirai2_vn_phase2'),
    enable_progress_bar=True,
    precision='16-mixed' if (DEVICE == 'cuda' and CFG.get('mixed_precision', False)) else '32'
)

trainer_p2.fit(model_p2, train_dataloaders=train_loader, val_dataloaders=val_loader)

print(f'Phase 2 best val_loss: {ckpt_p2.best_model_score}')
print(f'Phase 2 best ckpt: {ckpt_p2.best_model_path}')

c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
GPU available: True (cuda), 

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Moirai2Forecaster │  1.8 M │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.8 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 1.295
Epoch 0, global step 14: 'val_loss' reached 1.29478 (best 1.29478), saving model to 'C:\\Users\\ADMIN\\OneDrive - Hanoi University of Mining and Geology\\Documents\\NCKH\\TFT-GreenPower-Forecasting\\checkpoint\\moirai2_vn_v1_epoch=00_val_loss=1.2948.ckpt' as top 1


...

Epoch 8, global step 126: 'val_loss' was not in top 1


Epoch 9, global step 140: 'val_loss' was not in top 1


Epoch 10, global step 154: 'val_loss' was not in top 1


Monitored metric val_loss did not improve in the last 10 records. Best score: 1.294. Signaling Trainer to stop.
Epoch 11, global step 168: 'val_loss' was not in top 1


Phase 2 best val_loss: 1.2944254875183105
Phase 2 best ckpt: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\moirai2_vn_v1_epoch=01_val_loss=1.2944.ckpt


In [25]:
import shutil

FINAL_CKPT = CKPT_DIR / 'moirai2_vn_v1_best.ckpt'
FINAL_LATEST = CKPT_DIR / 'moirai2_vn_v1_latest.ckpt'
FINAL_WEIGHTS = CKPT_DIR / 'moirai2_vn_v1_weights.pt'
FINAL_CFG = CKPT_DIR / 'moirai2_vn_v1_config.json'

if ckpt_p2.best_model_path and os.path.exists(ckpt_p2.best_model_path):
    shutil.copy2(ckpt_p2.best_model_path, FINAL_CKPT)
if ckpt_p2.last_model_path and os.path.exists(ckpt_p2.last_model_path):
    shutil.copy2(ckpt_p2.last_model_path, FINAL_LATEST)

torch.save({
    'model_state_dict': model_p2.model.state_dict(),
    'feature_cols': feature_cols,
    'target_col': target_col,
    'cfg': CFG
}, FINAL_WEIGHTS)

vn_cfg = {
    'version': 'moirai2_vn_v1',
    'pretrained_from': str(PRETRAIN_WEIGHTS),
    'best_ckpt': str(FINAL_CKPT),
    'latest_ckpt': str(FINAL_LATEST),
    'weights': str(FINAL_WEIGHTS),
    'best_val_loss': float(ckpt_p2.best_model_score) if ckpt_p2.best_model_score is not None else None,
    'phase1_val_loss': float(ckpt_p1.best_model_score) if ckpt_p1.best_model_score is not None else None,
    'context_length': CFG['context_length'],
    'prediction_length': CFG['prediction_length'],
    'feature_cols': feature_cols,
    'quantiles': CFG['quantiles'],
    'fine_tune_series': sorted(df['series'].unique().tolist())
}

with open(FINAL_CFG, 'w', encoding='utf-8') as f:
    json.dump(vn_cfg, f, indent=2, ensure_ascii=False)

print(f'Final best ckpt: {FINAL_CKPT}')
print(f'Final latest ckpt: {FINAL_LATEST}')
print(f'Final config: {FINAL_CFG}')

Final best ckpt: checkpoint\moirai2_vn_v1_best.ckpt
Final latest ckpt: checkpoint\moirai2_vn_v1_latest.ckpt
Final config: checkpoint\moirai2_vn_v1_config.json


In [26]:
best_vn = LitMoirai2.load_from_checkpoint(str(FINAL_CKPT), cfg={**CFG, 'learning_rate': CFG['phase2_lr'], 'weight_decay': CFG['weight_decay']}, input_dim=input_dim)
best_vn.eval()
best_vn.to(DEVICE)

all_true = []
all_pred = []
median_idx = CFG['quantiles'].index(0.5) if 0.5 in CFG['quantiles'] else len(CFG['quantiles']) // 2

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        pred_q = best_vn(x)
        pred = pred_q[:, :, median_idx]
        all_true.append(y)
        all_pred.append(pred)

y_true = torch.cat(all_true, dim=0)
y_pred = torch.cat(all_pred, dim=0)
mae, rmse, smape, wape, r2 = compute_metrics(y_true, y_pred)

print('Validation metrics (VN transfer):')
print(f'  MAE   : {mae:.4f} TWh')
print(f'  RMSE  : {rmse:.4f} TWh')
print(f'  R2    : {r2:.4f}')
print(f'  SMAPE : {smape:.2f}%')
print(f'  WAPE  : {wape:.2f}%')

Validation metrics (VN transfer):
  MAE   : 3.8234 TWh
  RMSE  : 5.5520 TWh
  R2    : -0.2618
  SMAPE : 78.16%
  WAPE  : 72.83%


In [27]:
print('=' * 70)
print('MOIRAI-2 TRANSFER SUMMARY (VN)')
print('=' * 70)
print(f'Pretrained source : {PRETRAIN_WEIGHTS.name}')
print(f"Fine-tune series  : {sorted(df['series'].unique().tolist())}")
print(f'Phase 1 val_loss  : {ckpt_p1.best_model_score}')
print(f'Phase 2 val_loss  : {ckpt_p2.best_model_score}')
print(f'MAE   : {mae:.4f} TWh')
print(f'RMSE  : {rmse:.4f} TWh')
print(f'R2    : {r2:.4f}')
print(f'SMAPE : {smape:.2f}%')
print(f'WAPE  : {wape:.2f}%')
print(f'Checkpoint: {FINAL_CKPT}')
print(f'Config    : {FINAL_CFG}')
print('=' * 70)

MOIRAI-2 TRANSFER SUMMARY (VN)
Pretrained source : moirai2_v1_pretrain_weights.pt
Fine-tune series  : ['Coal', 'Gas', 'Hydro', 'Solar', 'Wind']
Phase 1 val_loss  : 1.2925317287445068
Phase 2 val_loss  : 1.2944254875183105
MAE   : 3.8234 TWh
RMSE  : 5.5520 TWh
R2    : -0.2618
SMAPE : 78.16%
WAPE  : 72.83%
Checkpoint: checkpoint\moirai2_vn_v1_best.ckpt
Config    : checkpoint\moirai2_vn_v1_config.json
